In [61]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor

In [62]:
housing = pd.read_csv("housing.csv")

In [63]:
housing['income_cat'] = pd.cut(housing['median_income'],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income_cat
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,5
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,5
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,5
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,4
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,3


In [64]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing['income_cat']):#
    strat_train_set = housing.loc[train_index] #It
    strat_test_set = housing.loc[test_index]
    strat_test_set.to_csv("input.csv", index=False)
    # Remove the income_cat column so the data is back to its original state
    for set_ in (strat_train_set, strat_test_set):
        set_.drop("income_cat", axis=1, inplace=True)
    strat_train_set.head()
    strat_test_set.head()

In [43]:
housing = strat_train_set.copy()


In [44]:
housing_features = housing.drop("median_house_value", axis=1)
housing_target = housing["median_house_value"].copy()

In [45]:
num_attributes = housing_features.drop("ocean_proximity", axis=1).columns.tolist()
cat_attributes = ["ocean_proximity"]

In [46]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('std_scaler', StandardScaler()),
])

In [47]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])


In [48]:
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attributes),
    ("cat", cat_pipeline, cat_attributes),
])

In [49]:
housing_prepared = full_pipeline.fit_transform(housing_features)

In [36]:
print(housing_prepared)

[[-0.94135046  1.34743822  0.02756357 ...  0.          0.
   0.        ]
 [ 1.17178212 -1.19243966 -1.72201763 ...  0.          0.
   1.        ]
 [ 0.26758118 -0.1259716   1.22045984 ...  0.          0.
   0.        ]
 ...
 [-1.5707942   1.31001828  1.53856552 ...  0.          0.
   0.        ]
 [-1.56080303  1.2492109  -1.1653327  ...  0.          0.
   0.        ]
 [-1.28105026  2.02567448 -0.13148926 ...  0.          0.
   0.        ]]


In [57]:
# Training the model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error


lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_target)
lin_pred=lin_reg.predict(housing_prepared)
lin_rmse = root_mean_squared_error(housing_target, lin_pred)
lin_rmses = -cross_val_score(lin_reg, housing_prepared, housing_target,
                             scoring="neg_root_mean_squared_error", cv=10)
# print("Linear Regression RMSE:", lin_rmse)
print("Linear Regression Cross-Validated RMSE:", pd.Series(lin_rmses).describe())


Linear Regression Cross-Validated RMSE: count       10.000000
mean     69204.322755
std       2500.382157
min      65318.224029
25%      67124.346106
50%      69404.658178
75%      70697.800632
max      73003.752739
dtype: float64


In [58]:
# using decision tree regressor
from sklearn.tree import DecisionTreeRegressor
tree_reg = DecisionTreeRegressor()
tree_reg.fit(housing_prepared,housing_target)
tree_pred = tree_reg.predict(housing_prepared)
tree_rmse = root_mean_squared_error(housing_target, tree_pred)
tree_rmses = -cross_val_score(tree_reg, housing_prepared, housing_target,
                              scoring="neg_root_mean_squared_error", cv=10)
#   print("Decision Tree RMSE:", tree_rmse)
print("Decision Tree Cross-Validated RMSE:", pd.Series(tree_rmses).describe())

Decision Tree Cross-Validated RMSE: count       10.000000
mean     69444.604990
std       2313.131280
min      64976.805168
25%      68764.713412
50%      69460.978019
75%      70133.667596
max      74043.488650
dtype: float64


In [59]:
#Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
forest_reg = RandomForestRegressor()
forest_reg.fit(housing_prepared, housing_target)
forest_pred = forest_reg.predict(housing_prepared)
forest_rmse = root_mean_squared_error(housing_target, forest_pred)
forest_rmses = -cross_val_score(forest_reg, housing_prepared, housing_target,
                                scoring="neg_root_mean_squared_error", cv=10)
# print("Random Forest RMSE:", forest_rmse)
print("Random Forest Cross-Validated RMSE:", pd.Series(forest_rmses).describe())

Random Forest Cross-Validated RMSE: count       10.000000
mean     49414.887072
std       2134.427171
min      45793.582552
25%      47945.926254
50%      49322.104383
75%      50920.483833
max      52835.288206
dtype: float64
